# Day 05 — Fabric R0/R1/R2 — DINOv3 ViT-S/16
Reuse FULL-v3 cache và cấu hình **notebook Day 04 của bạn**: r32/d384, fusion_dim64,
seed42, 150 epochs, batch64, AdamW lr0.001, AMP bfloat16, scheduler null, best by val_loss.

Code mới trong `day05_code_overlay.zip` phải được upload lên Drive trước cell 1.
Notebook kiểm tra SHA và không nhúng runner cũ ghi đè code. Không tải dataset/weights.

CODE PASS trên fixture không phải REAL-DATA PASS/FULL-TRAIN/FULL-EVALUATION COMPLETE.
Xem `docs/DAY05_PIPELINE_RUNBOOK.md` trong overlay cho hướng dẫn đầy đủ.

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, json, hashlib, zipfile, shutil

drive.mount('/content/drive', force_remount=False)
PROJECT_DRIVE = Path('/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9')
PROJECT_ROOT = Path('/content/msila')
OVERLAY = PROJECT_DRIVE / 'msila_day05/code/day05_code_overlay.zip'
OVERLAY_SHA256 = 'a2963dc169a0a9b0882c8f3f6078c513164f61841f2382e07e1fc2944900f194'
if not OVERLAY.is_file():
    raise FileNotFoundError(f'Upload reports/day05_code_overlay.zip vào {OVERLAY}')
if hashlib.sha256(OVERLAY.read_bytes()).hexdigest() != OVERLAY_SHA256:
    raise RuntimeError('Overlay SHA mismatch: dùng ZIP đi kèm notebook này')
if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(['git','clone','https://github.com/haduckien-raccoon/msila.git',str(PROJECT_ROOT)], check=True)
with zipfile.ZipFile(OVERLAY) as z:
    for entry in z.infolist():
        p = Path(entry.filename)
        if p.is_absolute() or '..' in p.parts:
            raise RuntimeError('Unsafe overlay entry')
    z.extractall(PROJECT_ROOT)
print('Code overlay verified:', OVERLAY_SHA256)

In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('BLOCKED_MISSING_GPU: chọn runtime GPU')
subprocess.run([sys.executable,'-m','pip','install','-q','pytest','pyyaml','scipy','pillow','matplotlib'],check=True)

In [ ]:
DAY04 = PROJECT_DRIVE / 'msila_day04'
DATA = DAY04 / 'data/full_screen_v3'
CACHE = DATA / 'feature_cache'
TRAIN = DATA / 'records/train_core.json'
DEV = DATA / 'records/dev_synthetic.json'
MASKS = DATA / 'masks'
DINO_REPO = Path('/content/dinov3')
DINO_CKPT = PROJECT_DRIVE / 'weights/dinov3_vits16_pretrain_lvd1689m.pth'
SELECTION = DAY04 / 'e8_full_v3/selection_report.json'
PROTOCOL = DAY04 / 'configs/full_v3/day04_full_seed_42.yaml'
if not PROTOCOL.is_file():
    PROTOCOL = PROJECT_ROOT / 'configs/day05_day04_full_v3_protocol.yaml'
    print('Dùng bản protocol trích từ cells 7,28 notebook Day 04:', PROTOCOL)
OUT = PROJECT_DRIVE / 'outputs/day05/full_train'
# Sửa theo RAW_ROOT/CATEGORY_LOCAL_ROOTS thật; cần raw cho DEV replay.
RAW = Path('/content/msila_data/mvtec_ad_2')
DEV_EXPORT = PROJECT_DRIVE / 'msila_day05/native_dev'
INPUTS = DEV_EXPORT / 'dev_inputs.json'
TINY = OUT / 'protocols/tiny_protocol.json'
BOUNDARY = OUT / 'protocols/boundary_protocol.json'

COMMON = ['--cache-dir',str(CACHE),'--train-records',str(TRAIN),'--val-records',str(DEV),
    '--mask-root',str(MASKS),'--dinov3-repo',str(DINO_REPO),'--dino-checkpoint',str(DINO_CKPT),
    '--adapter-selection-report',str(SELECTION),'--training-protocol',str(PROTOCOL),
    '--output-root',str(OUT),'--category','fabric','--seed','42','--device','cuda:0','--resume']
def run(module, *args):
    cmd = [sys.executable,'-m',module,*map(str,args)]
    print(subprocess.list2cmdline(cmd))
    subprocess.run(cmd,cwd=PROJECT_ROOT,check=True)

## Chuẩn bị tiny/boundary trước khi xem kết quả candidate
Day 04 không sinh hai file này. Skeleton trong repo có `null/false`, chưa phải scientific lock.
Cell dưới chỉ copy skeleton nếu file chưa tồn tại. Cần thống nhất `tiny_area_px`,
`band_width_px`, `tolerance_px` theo protocol/căn cứ GT độc lập với prediction và ghi basis.
Giữ threshold0.5 (cell1 Day04), connectivity8, FPR0.05. Chỉ xác nhận
`locked_before_candidate_results=true` khi đã khóa thật. Thiếu các giá trị này chỉ chặn
phần diagnostics/lock; không cần điền để chạy training hoặc inference.

In [ ]:
TINY.parent.mkdir(parents=True, exist_ok=True)
for source, target in [('day05_tiny_protocol.example.json',TINY),('day05_boundary_protocol.example.json',BOUNDARY)]:
    if not target.exists(): shutil.copyfile(PROJECT_ROOT/'configs'/source,target)
print('Protocol skeletons, chưa resolve:',TINY,BOUNDARY)

In [ ]:
run('scripts.run_day05_pipeline','--stage','audit','--dry-run',*COMMON)

In [ ]:
run('scripts.run_day05_pipeline','--stage','preflight',*COMMON)

## Full training — sequential R0 → R1 → R2
Cell sau chạy full 150 epochs mỗi candidate nếu assets/preflight PASS. `--resume` được bật.
Khi runtime ngắt, mount lại và chạy lại các cell setup rồi cell training. COMPLETE có checksum
sẽ được skip; last.pt khôi phục epoch/RNG, log epoch chưa commit được replay. Giữ nguyên paths,
protocol và code giữa các lần resume; không tự giảm batch hoặc đổi AMP để vượt gate.

In [ ]:
run('scripts.run_day05_pipeline','--stage','train',*COMMON)

## Replay DEV-synthetic native từ Day04 FULL-v3
Cache cũ chưa chứa ảnh synthetic native. Giữ đúng raw sources và đủ ba categories trong
records Day04. Export kiểm tra mask local và sáu DINO features trước khi xuất ảnh/mask native.
Nếu source/seed/generator/preprocessing khác, export FAIL và cần kiểm tra assets, không sửa nhãn backbone.

In [ ]:
run('scripts.build_day05_cache','--export-dev-only','--data-root',RAW,
    '--categories','fabric,vial,wallplugs','--cache-dir',CACHE,'--train-records',TRAIN,
    '--val-records',DEV,'--mask-root',MASKS,'--dinov3-repo',DINO_REPO,
    '--dino-checkpoint',DINO_CKPT,'--output-root',DEV_EXPORT,'--device','cuda:0')

In [ ]:
run('scripts.run_day05_pipeline','--stage','inference',*COMMON,'--input-manifest',INPUTS,'--seg-f1-threshold','0.5')

## Evaluation — cần hai protocol thực đã khóa
Reuse evaluator Day04 nguyên vẹn và chạy AU-PRO0.05 → tiny → boundary → region stats → comparison
→ efficiency → aggregation → representation lock. Không dùng TEST hoặc visualization chọn winner.
E8 cached head FP32/batch1/warmup10/50×3; native tiled runtime được lưu riêng từng image.
Các protocol skeleton còn null/false sẽ FAIL; không có CUDA thì E5/E6 BLOCKED.

In [ ]:
run('scripts.run_day05_pipeline','--stage','evaluate',*COMMON,'--input-manifest',INPUTS,
    '--seg-f1-threshold','0.5','--tiny-protocol',TINY,'--boundary-protocol',BOUNDARY)

In [ ]:
for candidate in ('R0','R1','R2'):
    p=OUT/'seed_42/fabric'/candidate/'run_manifest.json'
    if p.is_file():
        r=json.loads(p.read_text());print(candidate,r['status'],r.get('completed_epochs'),r.get('global_step'))
    else: print(candidate,'BLOCKED / chưa có run')
print('Native maps:', OUT/'maps/dev_synthetic')
print('Evaluation:', OUT/'evaluation/dev_synthetic')
print('Lock:', OUT/'evaluation/dev_synthetic/summary/representation_lock.yaml')